In [ ]:
import os
import pandas as pd
import yfinance as yf
import pyupbit
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. 초기 세팅
INITIAL_CAPITAL = 100000000
TARGET_WEIGHTS = {'BTC': 0.58, 'VOO': 0.145, 'ETH': 0.12, 'GOOGL': 0.055, 'USD': 0.10}

# 데이터 저장 폴더 세팅
DATA_DIR = './'

# 업비트 데이터 (BTC, ETH)
df_btc = pyupbit.get_ohlcv("KRW-BTC", interval="day", count=365)['close'].rename('BTC_KRW')
df_eth = pyupbit.get_ohlcv("KRW-ETH", interval="day", count=365)['close'].rename('ETH_KRW')

# 야후 파이낸스 데이터 (VOO, GOOGL, 환율)
yf_tickers = ['VOO', 'GOOGL', 'USDKRW=X']
df_yf = yf.download(yf_tickers, period="1y", interval="1d")['Close']
df_yf = df_yf.rename(columns={'USDKRW=X': 'USD'})

# 데이터 병합 및 주말 데이터 처리 (ffill)
df_merged = pd.concat([df_btc, df_eth, df_yf], axis=1)
df_merged[['VOO', 'GOOGL', 'USD']] = df_merged[['VOO', 'GOOGL', 'USD']].ffill()
df_merged = df_merged.dropna()

# 원화 환산 가격표
prices_krw = pd.DataFrame(index=df_merged.index)
prices_krw['BTC'] = df_merged['BTC_KRW']
prices_krw['ETH'] = df_merged['ETH_KRW']
prices_krw['VOO'] = df_merged['VOO'] * df_merged['USD']
prices_krw['GOOGL'] = df_merged['GOOGL'] * df_merged['USD']
prices_krw['USD'] = df_merged['USD']

# 3. 백테스트 지갑 세팅 (첫날 기준)
first_day = prices_krw.iloc[0]
wallet = {'BTC': 0, 'VOO': 0, 'ETH': 0, 'GOOGL': 0, 'USD': 0}
avg_buy_price = {'BTC': 0, 'VOO': 0, 'ETH': 0, 'GOOGL': 0}

# 벤치마크 수량 계산
btc_100_qty = INITIAL_CAPITAL / first_day['BTC']
voo_100_qty = INITIAL_CAPITAL / first_day['VOO']

for asset, weight in TARGET_WEIGHTS.items():
    allocated_krw = INITIAL_CAPITAL * weight
    wallet[asset] = allocated_krw / first_day[asset]
    if asset != 'USD':
        avg_buy_price[asset] = first_day[asset]

# 4. 시뮬레이션 루프
history = []
last_rebalance_month = None

for date, daily_prices in prices_krw.iterrows():
    # 현재 가치 평가
    current_values = {asset: wallet[asset] * daily_prices[asset] for asset in TARGET_WEIGHTS.keys()}
    total_value = sum(current_values.values())

    # 벤치마크 가치 평가
    btc_100_value = btc_100_qty * daily_prices['BTC']
    voo_100_value = voo_100_qty * daily_prices['VOO']

    # 리밸런싱 로직 (매월 9일)
    if date.day >= 9 and date.month != last_rebalance_month:
        last_rebalance_month = date.month

        for asset, target_w in TARGET_WEIGHTS.items():
            if asset == 'USD': continue
            current_w = current_values[asset] / total_value

            # 밴드 리밸런싱 (±20%)
            if current_w <= target_w * 0.8: # BUY
                required_krw = total_value * (target_w - current_w)
                if current_values['USD'] >= required_krw:
                    qty_to_buy = required_krw / daily_prices[asset]
                    wallet[asset] += qty_to_buy
                    wallet['USD'] -= required_krw / daily_prices['USD']

            elif current_w >= target_w * 1.2: # SELL + Profit Guard
                if daily_prices[asset] > avg_buy_price[asset]:
                    excess_krw = total_value * (current_w - target_w)
                    wallet[asset] -= excess_krw / daily_prices[asset]
                    wallet['USD'] += excess_krw / daily_prices['USD']

    history.append({
        'Date': date.strftime('%Y-%m-%d'),
        'Portfolio_Value': round(total_value, 2),
        'BTC_100_Value': round(btc_100_value, 2),
        'VOO_100_Value': round(voo_100_value, 2)
    })

# 5. 결과 분석 및 지표 산출
result_df = pd.DataFrame(history).set_index('Date')

def get_metrics(series):
    returns = series.pct_change()
    cum_return = (series.iloc[-1] / INITIAL_CAPITAL - 1) * 100
    mdd = ((series / series.cummax()) - 1).min() * 100
    sharpe = (returns.mean() / returns.std()) * np.sqrt(365)
    return round(cum_return, 2), round(mdd, 2), round(sharpe, 2)

p_ret, p_mdd, p_sha = get_metrics(result_df['Portfolio_Value'])
b_ret, b_mdd, b_sha = get_metrics(result_df['BTC_100_Value'])
v_ret, v_mdd, v_sha = get_metrics(result_df['VOO_100_Value'])

# 6. CSV 저장 (태블로용)
metrics_df = pd.DataFrame({
    'Strategy': ['Portfolio', 'BTC_Hold', 'VOO_Hold'],
    'Return_Pct': [p_ret, b_ret, v_ret],
    'MDD_Pct': [p_mdd, b_mdd, v_mdd],
    'Sharpe': [p_sha, b_sha, v_sha]
})

metrics_df.to_csv(os.path.join(DATA_DIR, 'backtest_summary_metrics.csv'), index=False, encoding='utf-8-sig')
result_df[['Portfolio_Value', 'BTC_100_Value', 'VOO_100_Value']].to_csv(os.path.join(DATA_DIR, 'backtest_daily_series.csv'), encoding='utf-8-sig')
